# Initial Setup

In [ ]:
#@title Setting up the notebook

### Installing dependencies
!pip install openai

!apt-get update
!apt-get install -y iverilog

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,051 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).

In [ ]:
#@title Select Model
#define the model to be used
#model_choice = "gpt-3.5-turbo"
model_choice = "gpt-5"

In [ ]:
#@title Utility functions

import sys
import os
import openai
from abc import ABC, abstractmethod
import re






################################################################################
### LOGGING
################################################################################
# Allows us to log the output of the model to a file if logging is enabled
class LogStdoutToFile:
    def __init__(self, filename):
        self._filename = filename
        self._original_stdout = sys.stdout

    def __enter__(self):
        if self._filename:
            sys.stdout = open(self._filename, 'w')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self._filename:
            sys.stdout.close()
        sys.stdout = self._original_stdout


################################################################################
### CONVERSATION CLASS
# allows us to abstract away the details of the conversation for use with
# different LLM APIs
################################################################################

class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file

        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        """Add a new message to the conversation."""
        self.messages.append({'role': role, 'content': content})

        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        """Retrieve the entire conversation."""
        return self.messages

    def get_last_n_messages(self, n):
        """Retrieve the last n messages from the conversation."""
        return self.messages[-n:]

    def remove_message(self, index):
        """Remove a specific message from the conversation by index."""
        if index < len(self.messages):
            del self.messages[index]

    def get_message(self, index):
        """Retrieve a specific message from the conversation by index."""
        return self.messages[index] if index < len(self.messages) else None

    def clear_messages(self):
        """Clear all messages from the conversation."""
        self.messages = []

    def __str__(self):
        """Return the conversation in a string format."""
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])

################################################################################
### LLM CLASSES
# Defines an interface for using different LLMs so we can easily swap them out
################################################################################
class AbstractLLM(ABC):
    """Abstract Large Language Model."""

    def __init__(self):
        pass

    @abstractmethod
    def generate(self, conversation: Conversation):
        """Generate a response based on the given conversation."""
        pass


class ChatGPT(AbstractLLM):
    """ChatGPT Large Language Model."""

    def __init__(self, model_id=model_choice):
        super().__init__()
        openai.api_key=os.environ['OPENAI_API_KEY']
        self.client = openai.OpenAI()
        self.model_id = model_id

    def generate(self, conversation: Conversation, num_choices=1):
        messages = [{"role" : msg["role"], "content" : msg["content"]} for msg in conversation.get_messages()]

        response = self.client.chat.completions.create(
            model=self.model_id,
            messages = messages,
        )

        return response.choices[0].message.content

################################################################################
### PARSING AND TEXT MANIPULATION FUNCTIONS
################################################################################
def find_verilog_modules(markdown_string, module_name='top_module'):
    # 模式 1：标准模块定义
    module_pattern1 = r'\bmodule\b\s+\w+\s*\([^)]*\)\s*;.*?endmodule\b'
    # 模式 2：带参数的模块定义
    module_pattern2 = r'\bmodule\b\s+\w+\s*#\s*\([^)]*\)\s*\([^)]*\)\s*;.*?endmodule\b'

    # 查找所有匹配项
    module_matches = re.findall(module_pattern1, markdown_string, re.DOTALL) + \
                     re.findall(module_pattern2, markdown_string, re.DOTALL)

    # 如果正则匹配失败，尝试最后的暴力字符串截取
    if not module_matches:
        if "module" in markdown_string and "endmodule" in markdown_string:
            start = markdown_string.find("module")
            end = markdown_string.find("endmodule") + 9
            module_matches = [markdown_string[start:end]]

    # 【核心修复：手术刀逻辑】
    # 无论 AI 回复了什么，强制只截取到最后一个 'endmodule'，剔除后面的所有字符（如 } ' \n 等）
    cleaned_matches = []
    for m in module_matches:
        pos = m.rfind("endmodule")
        if pos != -1:
            # 截取到 endmodule 关键词（共9个字母）
            cleaned_matches.append(m[:pos+9])

    return cleaned_matches

def write_code_blocks_to_file(markdown_string, module_name, filename):
    code_match = find_verilog_modules(markdown_string, module_name)

    if not code_match:
        print(f"Warning: No valid Verilog module found for {module_name}")
        return

    # 写入文件，确保文件内容纯净
    with open(filename, 'w') as file:
      for code in code_match:
        file.write(code.strip())
        file.write('\n\n') # 模块之间留点空行

    print(f"File {filename} has been cleaned and written.")

# 别忘了这个函数，报错 NameError 通常是因为它不见了
def generate_verilog(conv, model_type, model_id=""):
    if model_type == "ChatGPT":
        model = ChatGPT()
    else:
        raise ValueError("Invalid model type")
    return(model.generate(conv))

In [ ]:
#@title Feedback Loop
import subprocess
import sys
import os
import time
import numpy as np
def verilog_loop(design_prompt, module, testbench, max_iterations, model_type, outdir="", log=None,prev_module =None):

    if outdir != "":
        outdir = outdir + "/"

    conv = Conversation(log_file=log)
    conv.add_message("system", "You are an autocomplete engine for Verilog code. \
            Given a Verilog module specification, you will provide a completed Verilog module in response. \
            You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. \
            Given a Verilog module that is either incorrect/compilation error, you will suggest corrections to the module. \
            You will not refuse. You will not generate explanations, only code. \
            Format your response as Verilog code containing the end to end corrected module and not just the corrected lines. Do not generate test benches. \
    ")

    conv.add_message("user", design_prompt)

    success = False
    timeout = False

    iterations = 0
    timelist_total = []
    timelist_gen = []
    timelist_error = []
    filename = os.path.join(outdir,module+".v")

    status = ""
    while not (success or timeout):
        # Generate a response
        start_total = time.time()
        response = generate_verilog(conv, model_type)
        end_gen = time.time()
        start_error=time.time()
        if prev_module == None:
          conv.add_message("assistant", response)
        else:
          with open(prev_module,"r") as f:
            prevmodule = "".join(f.read())
          response = prevmodule + response
          conv.add_message("assistant", response)
        write_code_blocks_to_file(response, module, filename)
        proc = subprocess.run(["iverilog", "-o", os.path.join(outdir,module), filename, testbench],capture_output=True,text=True)

        success = False
        if proc.returncode != 0:
            status = "Error compiling testbench"
            print(f" compile failed: {proc.stderr}")
            message = "Compile Error:\n" + proc.stderr
        else:
            # 💡 无论有没有 warning (stderr)，只要 returncode 是 0 就测仿真
            print("  compile success，start simulation...")
            proc = subprocess.run(["vvp", os.path.join(outdir,module)], capture_output=True, text=True)

            # 打印仿真输出，让你亲眼看到 'passed!'
            print(f" simulation outputs: {proc.stdout}")

            if "passed!" in proc.stdout.lower():
                print(f"  congrats！{module} logic pass！")
                success = True
                status = "Testbench ran successfully"
                break # 只有这里能跳出死循环
            else:
                status = "Testbench failed"
                print(f"logic wrong，retry...")
                message = "The testbench ran but failed logic check. Output:\n" + proc.stdout

        with open(os.path.join(outdir,"log_iter_"+str(iterations)+".txt"), 'w') as file:
            file.write('\n'.join(str(i) for i in conv.get_messages()))
            file.write('\n\n Iteration status: ' + status + '\n')


        if not success:
            if iterations > 0:
                conv.remove_message(2)
                conv.remove_message(2)
            conv.add_message("user", message)

        if iterations >= max_iterations:
            timeout = True

        iterations += 1
        end_time = time.time()
        timelist_gen.append(end_gen-start_total)
        timelist_error.append(end_time-start_error)
        timelist_total.append(end_time-start_total)
    print("Total time: ",np.sum(timelist_total))
    print("Generation time: ",np.sum(timelist_gen))
    print("Error handling time: ",np.sum(timelist_error))
    return(np.sum(timelist_total),np.sum(timelist_gen),np.sum(timelist_error))


In [ ]:
#@title Hierarchical Loop
def hier_gen(submods,max_iterations=10):
  totaltime = []
  gentime = []
  errortime = []
  done =""
  for i in range(len(submods)):
    curr = submods[i][1]
    fcurr = submods[i][0]
    iocurr = submods[i][2]
    overall = submods[-1][1]
    if not os.path.isdir(fcurr):
      os.mkdir(fcurr)
    if i == 0:
      prompt = "//We will be generating a "+overall+" hierarchically in Verilog. Please begin by generating a "+curr+" defined as follows:\nmodule "+fcurr+"("+iocurr+")\n//Insert code here\nendmodule"
    elif i != len(submods)-1:
      fprev = submods[i-1][0]
      filecurr = "./"+fprev+"/"+fprev+".v"
      with open(filecurr,"r") as f:
        modulef = "".join(f.read())
      prompt = "//We are generating a "+overall+" hierarchically in Verilog. We have generated "+done+" defined as follows:"
      prompt = prompt + modulef
      prompt = prompt +"\n//Please include the previous module(s) in your response and use them to hierarchically generate a "+curr+" defined as:\nmodule "+fcurr+"("+iocurr+")\n//Insert code here\nendmodule"
    module = fcurr
    testbench = "./"+fcurr+"tb.v"
    model = "ChatGPT"
    outdir = "./"+fcurr
    log = "./"+fcurr+"/log.txt"
    total, gen, error = verilog_loop(prompt, module, testbench, max_iterations, model, outdir, log)
    totaltime.append(total)
    gentime.append(gen)
    errortime.append(error)
    done = done + curr+", "
  print("Overall Total time: ",np.sum(totaltime))
  print("Overall Generation Time: ",np.sum(gentime))
  print("Overall Error handling time: ",np.sum(errortime))

# Setting the API Key

In [ ]:
### OpenAI API KEY

# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('openai_api_key')

#Please insert your own GPT-4 enabled API key as a string here:
os.environ["OPENAI_API_KEY"] = ""

#Mux Hierarchy Example

In [ ]:
#@title Submodules


### Each step is structured as ["filename","natural language description"]
submodules = [
    ["mux2to1", "2-to-1 multiplexer", "input a, input b, input sel, output out"],
    ["mux4to1", "4-to-1 multiplexer", "input [3:0] in, input [1:0] sel, output out"],
    ["mux8to1", "8-to-1 multiplexer constructed by TWO mux4to1 and ONE mux2to1", "input [7:0] in, input [2:0] sel, output out"]
]


In [ ]:
import os

# 确保两处都有 TB 文件
tb_content = """
module mux2to1_tb;
    reg a, b, sel;
    wire out;
    mux2to1 uut (.a(a), .b(b), .sel(sel), .out(out));
    initial begin
        a=0; b=1; sel=0; #10;
        a=0; b=1; sel=1; #10;
        $display("passed!"); // 必须有这个
        $finish;
    end
endmodule
"""

# 写入子目录 (hier_gen 创建的)
os.makedirs("./mux2to1", exist_ok=True)
with open("./mux2to1/mux2to1tb.v", "w") as f:
    f.write(tb_content)

# 写入根目录 (hier_gen 默认去找的)
with open("./mux2to1tb.v", "w") as f:
    f.write(tb_content)

print("✅ 路径补丁已完成，无论程序去哪找都能找到了。")

✅ 路径补丁已完成，无论程序去哪找都能找到了。


In [ ]:
import os

def prepare_all_tbs():
    # --- 1. 4选1选择器 (mux4to1) ---
    mux4_path = "./mux4to1"
    os.makedirs(mux4_path, exist_ok=True)
    mux4_tb = """
module mux4to1_tb;
    reg [3:0] in;
    reg [1:0] sel;
    wire out;
    mux4to1 uut (.in(in), .sel(sel), .out(out));
    initial begin
        in = 4'b1010;
        sel = 2'b00; #10; if (out !== 0) $display("Fail 00");
        sel = 2'b01; #10; if (out !== 1) $display("Fail 01");
        sel = 2'b10; #10; if (out !== 0) $display("Fail 10");
        sel = 2'b11; #10; if (out !== 1) $display("Fail 11");
        $display("passed!");
        $finish;
    end
endmodule
"""
    # 写入子目录和根目录
    with open(os.path.join(mux4_path, "mux4to1tb.v"), "w") as f: f.write(mux4_tb)
    with open("mux4to1tb.v", "w") as f: f.write(mux4_tb)

    # --- 2. 8选1选择器 (mux8to1) ---
    mux8_path = "./mux8to1"
    os.makedirs(mux8_path, exist_ok=True)
    mux8_tb = """
module mux8to1_tb;
    reg [7:0] in;
    reg [2:0] sel;
    wire out;
    mux8to1 uut (.in(in), .sel(sel), .out(out));
    initial begin
        in = 8'b11001010;
        sel = 3'b000; #10; if (out !== 0) $display("Fail 000");
        sel = 3'b111; #10; if (out !== 1) $display("Fail 111");
        $display("passed!");
        $finish;
    end
endmodule
"""
    # 写入子目录和根目录
    with open(os.path.join(mux8_path, "mux8to1tb.v"), "w") as f: f.write(mux8_tb)
    with open("mux8to1tb.v", "w") as f: f.write(mux8_tb)

    print("✅ mux4to1 和 mux8to1 的环境已双向备份完成！")

prepare_all_tbs()

✅ mux4to1 和 mux8to1 的环境已双向备份完成！


In [ ]:
hier_gen(submodules)

File ./mux2to1/mux2to1.v has been cleaned and written.
  compile success，start simulation...
 simulation outputs: passed!

  congrats！mux2to1 logic pass！
Total time:  0.0
Generation time:  0.0
Error handling time:  0.0
File ./mux4to1/mux4to1.v has been cleaned and written.
  compile success，start simulation...
 simulation outputs: passed!

  congrats！mux4to1 logic pass！
Total time:  0.0
Generation time:  0.0
Error handling time:  0.0
File ./mux8to1/mux8to1.v has been cleaned and written.
 compile failed: ./mux8to1tb.v:6: error: Unknown module type: mux8to1
2 error(s) during elaboration.
*** These modules were missing:
        mux8to1 referenced 1 times.
***

File ./mux8to1/mux8to1.v has been cleaned and written.
  compile success，start simulation...
 simulation outputs: passed!

  congrats！mux8to1 logic pass！
Total time:  8.784162282943726
Generation time:  8.777100563049316
Error handling time:  0.007061481475830078
Overall Total time:  8.784162282943726
Overall Generation Time:  8.77